# EDA-6 — Redundancy and Construct Overlap Analysis

Orchestration notebook. Executes governed pairwise redundancy detection and construct-overlap analysis across all governed signals and positions.

**Outputs:**
- `research/eda/findings/eda_06_pairwise_rho.csv` — Spearman rho for every unique signal pair per position
- `research/eda/findings/eda_06_construct_map.csv` — flagged pairs annotated with relationship type
- `research/eda/findings/eda_06_partial_rho.csv` — partial rho controlling for target per flagged pair

**Depends on:** `analytics.signals.redundancy`, `pipeline.dal.prepared.analytical_dataset`

All redundancy logic is imported — no inline thresholds, construct heuristics, or correlation logic.

## Setup

In [1]:
from __future__ import annotations

from itertools import combinations
from pathlib import Path

import pandas as pd

from pipeline.config import DB_PATH
from dal.curated.player_gameweek_spine import build_player_gameweek_spine
from dal.prepared.analytical_dataset import (
    build_prepared_dataset,
    GOVERNED_SIGNAL_COLUMNS,
)
from core.signals.redundancy import (
    ALGEBRAIC_DECOMPOSITIONS,
    compute_pairwise_rho,
    compute_partial_rho,
    identify_redundant_pairs,
)

## Paths and constants

All output paths are defined here. `DATA_CUTOFF_GW` is the only notebook-local parameter.

In [2]:
FINDINGS_DIR = Path("../findings")
FINDINGS_DIR.mkdir(parents=True, exist_ok=True)

PAIRWISE_RHO_PATH  = FINDINGS_DIR / "eda_06_pairwise_rho.csv"
CONSTRUCT_MAP_PATH = FINDINGS_DIR / "eda_06_construct_map.csv"
PARTIAL_RHO_PATH   = FINDINGS_DIR / "eda_06_partial_rho.csv"

# Upper GW bound for the prepared dataset. Set to the last completed gameweek.
DATA_CUTOFF_GW: int = 38

# Target variable for partial correlation computation.
TARGET: str = "total_points"

# Governed positions — matches POSITION_CODE_MAP values in analytical_dataset.
POSITIONS: list[str] = ["GK", "DEF", "MID", "FWD"]

# Governed signal set — sourced directly from the prepared dataset contract.
SIGNALS: list[str] = list(GOVERNED_SIGNAL_COLUMNS)

# Governed relationship-type vocabulary — closed set.
RELATIONSHIP_TYPE_VALUES: frozenset[str] = frozenset({"algebraic_decomposition", "statistical_redundancy"})

print(f"DB path:         {DB_PATH}")
print(f"Data cutoff GW:  {DATA_CUTOFF_GW}")
print(f"Positions:       {POSITIONS}")
print(f"Signal count:    {len(SIGNALS)}")
print(f"Target:          {TARGET}")
print(f"Algebraic decompositions: {ALGEBRAIC_DECOMPOSITIONS}")

DB path:         /Users/safarifgisa/.fpl/fpl.db
Data cutoff GW:  38
Positions:       ['GK', 'DEF', 'MID', 'FWD']
Signal count:    29
Target:          total_points
Algebraic decompositions: (('xgi', 'xg', 'xa'), ('ict_index', 'influence', 'creativity'))


## 1. Load prepared analytical dataset

Builds the governed prepared dataset from the curated spine. Fails early if required columns are missing.

In [3]:
print(f"Loading spine from {DB_PATH} ...")
spine = build_player_gameweek_spine(DB_PATH)

print(f"Building prepared dataset (cutoff GW={DATA_CUTOFF_GW}) ...")
prepared = build_prepared_dataset(spine, data_cutoff_gw=DATA_CUTOFF_GW)

print(f"Prepared dataset: {len(prepared):,} rows × {len(prepared.columns)} columns")
print(f"GW range: {prepared['gw'].min()} – {prepared['gw'].max()}")
print(f"Position distribution:\n{prepared['position'].value_counts().to_string()}")

Loading spine from /Users/safarifgisa/.fpl/fpl.db ...
Building prepared dataset (cutoff GW=38) ...
Prepared dataset: 7,111 rows × 33 columns
GW range: 1 – 35
Position distribution:
position
MID    2996
DEF    2744
GK      687
FWD     684


## 2. Validate required columns

Asserts all governed signals and structural columns are present before any analysis executes.

In [4]:
REQUIRED_COLUMNS = {"position", TARGET} | set(SIGNALS)
missing_columns = REQUIRED_COLUMNS - set(prepared.columns)
if missing_columns:
    raise ValueError(
        f"Prepared dataset is missing required columns: {sorted(missing_columns)}\n"
        "Cannot proceed with redundancy analysis."
    )

# Filter signals to those actually present (some may be absent for valid reasons).
signals_present = [s for s in SIGNALS if s in prepared.columns]
signals_absent  = [s for s in SIGNALS if s not in prepared.columns]

if signals_absent:
    print(f"WARNING: {len(signals_absent)} governed signals not in prepared dataset: {signals_absent}")

# Cast pandas ExtensionDtype signal columns (Int64, Float64, boolean) to float64.
# scipy.stats.spearmanr inside compute_pairwise_rho requires standard numpy dtypes.
prepared = prepared.copy()
for col in signals_present + [TARGET]:
    if col in prepared.columns and hasattr(prepared[col].dtype, "numpy_dtype"):
        prepared[col] = prepared[col].astype(float)

print(f"Column validation passed. Signals to analyze: {len(signals_present)}")

Column validation passed. Signals to analyze: 29


## 3. Execute governed redundancy analysis

For each position:
1. Compute pairwise Spearman rho matrix via `compute_pairwise_rho`.
2. Extract upper-triangle pairs (no self-correlations, no mirrored duplicates) into long form.
3. Identify statistically redundant pairs via `identify_redundant_pairs`.

No inline correlation heuristics or redundancy thresholds.

In [5]:
pairwise_rows: list[dict] = []
# Redundant pairs indexed by position for construct map assembly.
redundant_pairs_by_position: dict[str, list[tuple[str, str]]] = {}

print("Computing pairwise rho matrices ...")
for position in POSITIONS:
    rho_matrix = compute_pairwise_rho(
        df=prepared,
        signals=signals_present,
        position=position,
    )

    # Extract upper triangle only: no self-pairs, no mirrored duplicates.
    signals_in_matrix = list(rho_matrix.columns)
    for i, j in combinations(range(len(signals_in_matrix)), 2):
        sig_a = signals_in_matrix[i]
        sig_b = signals_in_matrix[j]
        val = rho_matrix.iloc[i, j]
        # Emit in lexicographic order for deterministic pair identity.
        sig_a_out, sig_b_out = (sig_a, sig_b) if sig_a <= sig_b else (sig_b, sig_a)
        pairwise_rows.append({
            "signal_a": sig_a_out,
            "signal_b": sig_b_out,
            "position": position,
            "rho": val if pd.notna(val) else None,
        })

    redundant_pairs_by_position[position] = identify_redundant_pairs(rho_matrix)
    print(f"  {position}: {len(redundant_pairs_by_position[position])} redundant pair(s)")

pairwise_df = pd.DataFrame(pairwise_rows)
print(f"\nPairwise rho rows: {len(pairwise_df):,}")

Computing pairwise rho matrices ...
  GK: 7 redundant pair(s)
  DEF: 4 redundant pair(s)
  MID: 4 redundant pair(s)
  FWD: 6 redundant pair(s)

Pairwise rho rows: 1,624


## 4. Build construct map

Combines algebraic decomposition pairs (from `ALGEBRAIC_DECOMPOSITIONS`) with statistically redundant pairs (from `identify_redundant_pairs`). Algebraic pairs take precedence in relationship_type assignment.

In [6]:
# Build the governed set of algebraic pairs present in the signal set.
# For each (derived, comp_a, comp_b) in ALGEBRAIC_DECOMPOSITIONS, the
# relevant pairings are (derived, comp_a) and (derived, comp_b).
algebraic_pairs: set[tuple[str, str]] = set()
for derived, comp_a, comp_b in ALGEBRAIC_DECOMPOSITIONS:
    sig_set = set(signals_present)
    if derived in sig_set and comp_a in sig_set:
        algebraic_pairs.add(tuple(sorted((derived, comp_a))))
    if derived in sig_set and comp_b in sig_set:
        algebraic_pairs.add(tuple(sorted((derived, comp_b))))

construct_rows: list[dict] = []

for position in POSITIONS:
    # Rho lookup for this position.
    pos_rho = pairwise_df[pairwise_df["position"] == position].set_index(["signal_a", "signal_b"])["rho"]

    def _lookup_rho(a: str, b: str) -> float | None:
        key = tuple(sorted((a, b)))
        return pos_rho.get(key, None)

    # Algebraic decomposition pairs (position-independent by definition, but
    # rho is position-specific so emit one row per position).
    for sig_a, sig_b in sorted(algebraic_pairs):
        construct_rows.append({
            "signal_a": sig_a,
            "signal_b": sig_b,
            "position": position,
            "relationship_type": "algebraic_decomposition",
            "rho": _lookup_rho(sig_a, sig_b),
        })

    # Statistical redundancy pairs — exclude algebraic pairs to avoid double-counting.
    for sig_a, sig_b in redundant_pairs_by_position[position]:
        pair_key = tuple(sorted((sig_a, sig_b)))
        if pair_key in algebraic_pairs:
            continue
        construct_rows.append({
            "signal_a": min(sig_a, sig_b),
            "signal_b": max(sig_a, sig_b),
            "position": position,
            "relationship_type": "statistical_redundancy",
            "rho": _lookup_rho(sig_a, sig_b),
        })

construct_df = pd.DataFrame(construct_rows)
print(f"Construct map rows: {len(construct_df)}")
if len(construct_df) > 0:
    print(construct_df["relationship_type"].value_counts().to_string())

Construct map rows: 33
relationship_type
statistical_redundancy     17
algebraic_decomposition    16


## 5. Compute partial rho for flagged pairs

Calls `compute_partial_rho` for each unique (signal_a, signal_b, position) pair in the construct map, controlling for `total_points`.

In [7]:
partial_rho_rows: list[dict] = []

print("Computing partial rho for flagged pairs ...")
for _, row in construct_df[["signal_a", "signal_b", "position"]].drop_duplicates().iterrows():
    sig_a    = row["signal_a"]
    sig_b    = row["signal_b"]
    position = row["position"]

    partial = compute_partial_rho(
        df=prepared,
        signal_a=sig_a,
        signal_b=sig_b,
        target=TARGET,
        position=position,
    )

    partial_rho_rows.append({
        "signal_a":   sig_a,
        "signal_b":   sig_b,
        "target":     TARGET,
        "position":   position,
        "partial_rho": partial,
    })

partial_rho_df = pd.DataFrame(partial_rho_rows)
print(f"Partial rho rows: {len(partial_rho_df)}")

Computing partial rho for flagged pairs ...
Partial rho rows: 33


## 6. Sort outputs for deterministic writes

In [8]:
pairwise_df = pairwise_df.sort_values(["position", "signal_a", "signal_b"]).reset_index(drop=True)
construct_df = construct_df.sort_values(["position", "relationship_type", "signal_a", "signal_b"]).reset_index(drop=True)
partial_rho_df = partial_rho_df.sort_values(["position", "signal_a", "signal_b"]).reset_index(drop=True)

print(f"pairwise_df:    {len(pairwise_df)} rows × {list(pairwise_df.columns)}")
print(f"construct_df:   {len(construct_df)} rows × {list(construct_df.columns)}")
print(f"partial_rho_df: {len(partial_rho_df)} rows × {list(partial_rho_df.columns)}")

pairwise_df:    1624 rows × ['signal_a', 'signal_b', 'position', 'rho']
construct_df:   33 rows × ['signal_a', 'signal_b', 'position', 'relationship_type', 'rho']
partial_rho_df: 33 rows × ['signal_a', 'signal_b', 'target', 'position', 'partial_rho']


## 7. Validate outputs before writing

In [9]:
# Assert no null signal names across all outputs.
for df_name, df_check, cols in [
    ("pairwise_df",    pairwise_df,    ["signal_a", "signal_b"]),
    ("construct_df",   construct_df,   ["signal_a", "signal_b"]),
    ("partial_rho_df", partial_rho_df, ["signal_a", "signal_b"]),
]:
    for col in cols:
        nulls = df_check[col].isna().sum()
        assert nulls == 0, f"{df_name}.{col} has {nulls} null values"

# Assert no duplicate (signal_a, signal_b, position) rows in pairwise_df.
pairwise_dups = int(pairwise_df[["signal_a", "signal_b", "position"]].duplicated().sum())
assert pairwise_dups == 0, f"pairwise_df has {pairwise_dups} duplicate (signal_a, signal_b, position) rows"

# Assert no duplicate (signal_a, signal_b, position) rows in construct_df.
construct_dups = int(construct_df[["signal_a", "signal_b", "position"]].duplicated().sum())
assert construct_dups == 0, f"construct_df has {construct_dups} duplicate (signal_a, signal_b, position) rows"

# Assert rho values in pairwise_df are bounded in [-1, 1] or NaN.
non_nan_rho = pairwise_df["rho"].dropna()
rho_oob = ((non_nan_rho < -1) | (non_nan_rho > 1)).sum()
assert rho_oob == 0, f"pairwise_df has {rho_oob} rho values outside [-1, 1]"

# Assert partial_rho values are bounded in [-1, 1] or NaN.
non_nan_partial = partial_rho_df["partial_rho"].dropna()
partial_oob = ((non_nan_partial < -1) | (non_nan_partial > 1)).sum()
assert partial_oob == 0, f"partial_rho_df has {partial_oob} partial_rho values outside [-1, 1]"

# Assert algebraic decomposition rows come only from ALGEBRAIC_DECOMPOSITIONS.
alg_rows = construct_df[construct_df["relationship_type"] == "algebraic_decomposition"]
governed_alg_pairs = {
    tuple(sorted((d, a)))
    for d, a, b in ALGEBRAIC_DECOMPOSITIONS
    for a_or_b in (a, b)
    for pair in [tuple(sorted((d, a_or_b)))]
}
spurious_alg = [
    (r["signal_a"], r["signal_b"])
    for _, r in alg_rows.iterrows()
    if tuple(sorted((r["signal_a"], r["signal_b"]))) not in algebraic_pairs
]
assert not spurious_alg, (
    f"construct_df contains algebraic_decomposition rows not in ALGEBRAIC_DECOMPOSITIONS: {spurious_alg}"
)

# Assert relationship_type vocabulary is governed and closed.
invalid_rel_types = set(construct_df["relationship_type"].unique()) - RELATIONSHIP_TYPE_VALUES
assert not invalid_rel_types, (
    f"construct_df contains unrecognized relationship_type values: {invalid_rel_types}\n"
    f"Governed vocabulary: {sorted(RELATIONSHIP_TYPE_VALUES)}"
)

print("All output validations passed.")

All output validations passed.


## 8. Write governed outputs

In [10]:
pairwise_df.to_csv(PAIRWISE_RHO_PATH, index=False)
print(f"Written: {PAIRWISE_RHO_PATH}  ({len(pairwise_df)} rows)")

construct_df.to_csv(CONSTRUCT_MAP_PATH, index=False)
print(f"Written: {CONSTRUCT_MAP_PATH}  ({len(construct_df)} rows)")

partial_rho_df.to_csv(PARTIAL_RHO_PATH, index=False)
print(f"Written: {PARTIAL_RHO_PATH}  ({len(partial_rho_df)} rows)")

Written: ../findings/eda_06_pairwise_rho.csv  (1624 rows)
Written: ../findings/eda_06_construct_map.csv  (33 rows)
Written: ../findings/eda_06_partial_rho.csv  (33 rows)


## 9. Summary

Summary counts only. No speculative interpretation.

In [11]:
print("=" * 60)
print("EDA-6 — Redundancy Analysis Summary")
print("=" * 60)

if len(construct_df) > 0:
    print("\nRedundancy counts by position:")
    pos_counts = construct_df.groupby("position")["relationship_type"].value_counts().unstack(fill_value=0)
    print(pos_counts.to_string())

    print("\nAlgebraic vs statistical redundancy counts:")
    rel_counts = construct_df["relationship_type"].value_counts().reindex(
        sorted(RELATIONSHIP_TYPE_VALUES), fill_value=0
    )
    for label, count in rel_counts.items():
        print(f"  {label:<30} {count}")
else:
    print("\nNo flagged pairs found at governed redundancy threshold.")

print("\nTop 10 absolute rho pairs (all positions):")
non_self = pairwise_df[pairwise_df["rho"].notna()].copy()
non_self["abs_rho"] = non_self["rho"].abs()
top_pairs = non_self.nlargest(10, "abs_rho")[["signal_a", "signal_b", "position", "rho"]]
print(top_pairs.to_string(index=False))

print("\nOutputs:")
print(f"  {PAIRWISE_RHO_PATH}")
print(f"  {CONSTRUCT_MAP_PATH}")
print(f"  {PARTIAL_RHO_PATH}")
print("=" * 60)

EDA-6 — Redundancy Analysis Summary

Redundancy counts by position:
relationship_type  algebraic_decomposition  statistical_redundancy
position                                                          
DEF                                      4                       4
FWD                                      4                       4
GK                                       4                       5
MID                                      4                       4

Algebraic vs statistical redundancy counts:
  algebraic_decomposition        16
  statistical_redundancy         17

Top 10 absolute rho pairs (all positions):
signal_a signal_b position  rho
 fdr_avg  fdr_max      DEF  1.0
 fdr_avg  fdr_min      DEF  1.0
 fdr_max  fdr_min      DEF  1.0
 fdr_avg  fdr_max      FWD  1.0
 fdr_avg  fdr_min      FWD  1.0
 fdr_max  fdr_min      FWD  1.0
 fdr_avg  fdr_max       GK  1.0
 fdr_avg  fdr_min       GK  1.0
 fdr_max  fdr_min       GK  1.0
 fdr_avg  fdr_max      MID  1.0

Outputs:
  ../fi